
Graph-State Encrypted Cloning Certification (GSECC)
and Graph-State Decoder Construction (GSDC)
====================================================

Both problems are parameterized explicitly by (m, k), matching the
theoretical setup:

    k = number of input (signal) qubits being encrypted-clone-protected
    m = number of clones
    mk = m * k, the size of each half of the balanced bipartition
    2mk = |V|, the required number of vertices in the graph

PROBLEM 1 -- GSECC (certification)
-----------------------------------
GSECC is an EXACT decision problem:

    Given a graph G = (V, E) and integers m, k, does there exist a
    balanced partition V = S ∪ N, |S| = |N| = mk, such that

        rank_GF(2)(Gamma_{S,N}) = mk,

    where Gamma_{S,N} is the biadjacency (cut) matrix connecting S to N?

Algorithm:
    Step 1. Verify |V| = 2mk. If not, reject immediately (ValueError).
    Step 2. Enumerate every balanced partition (S, N).
    Step 3. For each, compute Gamma_{S,N}.
    Step 4. Compute rank_GF(2)(Gamma_{S,N}) via Gaussian elimination.
    Step 5. If rank = mk, return "Certified" with witness (S, N).
    Step 6. If every partition has been tested and none certifies,
            return "No valid balanced partition exists" -- a proof of
            non-existence, not a heuristic failure.

If a certificate exists, the graph state is certified as a valid
encrypted cloning resource with reduced state on S maximally mixed:

    rho_S = I / 2^{mk}.

Complexity:
    * Verification of a single candidate partition: O((mk)^3), via GF(2)
      Gaussian elimination on Gamma_{S,N}.
    * Search: the algorithm enumerates all C(2mk-1, mk-1) balanced
      partitions (one vertex is fixed to remove the S<->N labeling
      symmetry), so the overall search is
          O( C(2mk-1, mk-1) * (mk)^3 ).
    * This is exact, not approximate: every "Certified" and every "No
      valid balanced partition exists" outcome is a mathematically
      proven statement, not a heuristic result. The only practical
      limitation is scalability of the exhaustive search for large mk
      (e.g. mk=10 -> 92,378 partitions; mk=20 -> ~6.9e10 partitions);
      that is a computational limitation of exhaustive search, not a
      flaw in the certification algorithm itself.

This script implements only the exact algorithm above
(find_certificate_exact). An earlier version also included a randomized
search strategy for exploring very large graphs; that heuristic has been
removed here because GSECC is presented as an exact certification
algorithm, and a randomized search would only produce inconclusive
negative results ("not found within budget" rather than "does not
exist"), which weakens that presentation. If very large instances need to
be explored heuristically, that is a separate, clearly-labeled
experimental tool -- not part of the formal certification algorithm.

rho_S is never printed as a matrix -- only the boolean check

    || rho_S - I/2^{mk} ||  ==  0   (up to numerical tolerance)

and the residual norm are reported.


PROBLEM 2 -- GSDC (decoder construction)
------------------------------------------
Given a certified partition (S, N) from GSECC (for the same m, k), construct
the unitary W : H_N -> H_N such that

    |G> = (I_S ⊗ W) |Phi_{2^{mk}}>,

where |Phi_{2^{mk}}> = (1/sqrt(2^{mk})) sum_i |i>_S |i>_N is the canonical
maximally entangled state. Construction:

    1. Reorder qubits so S-qubits precede N-qubits.
    2. Write |G> = sum_{i,j} M_{ij} |i>_S |j>_N  (M is 2^{mk} x 2^{mk}).
    3. Certification guarantees rho_S = I/2^{mk}, so Q = sqrt(2^{mk}) * M
       is unitary.
    4. W = Q^T.

Polynomial in the Hilbert-space dimension d = 2^{mk} (reshaping, building
Q, transposing are all O(d^2)); exponential in the qubit number, as
expected for explicit state-vector manipulation.

This script provides:
  1. GF(2) linear algebra (rank / invertibility via Gaussian elimination).
  2. validate_graph_size(A, m, k)        -> enforce |V| = 2mk, return mk.
  3. verify_certificate(A, S, N, m, k)   -> poly-time GSECC verifier.
  4. find_certificate_exact(A, m, k)     -> the exact GSECC algorithm:
                                             exhaustive search over every
                                             balanced bipartition. A
                                             return value of None is a
                                             proof of non-existence.
  5. graph_state_from_adjacency(A)       -> build |G> from a graph.
  6. reduced_density_matrix(state,...)   -> rho_S from a state vector.
  7. rho_S_matches_maximally_mixed(...)  -> norm-based pass/fail check.
  8. construct_decoder(psi, S, N, m, k)  -> GSDC: builds W and verifies
                                             |G> = (I_S ⊗ W)|Phi_{2^{mk}}>.
  9. print_parameters(m, k, A)           -> prints the (m, k, mk, |V|)
                                             header for a run.
 10. plot_graph(A, S, N, title)          -> publication-quality figure:
                                             two-column S|N layout with the
                                             cut edges (Gamma_{S,N}) drawn
                                             in blue when a certificate
                                             exists, or an ordinary spring
                                             layout when it doesn't. Saves
                                             a PDF + 600 dpi PNG per graph.
 11. report_certificate(...)             -> shared reporting routine, keyed
                                             on (m, k) throughout, calling
                                             plot_graph for every example
                                             and GSDC automatically whenever
                                             a certificate exists. Since
                                             find_certificate_exact() is
                                             exhaustive, a None result is
                                             always reported as a proven
                                             negative: "No valid balanced
                                             partition exists."
 12. bell_pair_matching_graph / star_graph / path_graph / grid_graph /
     ghz_state helpers, plus a demo covering K6, C8, P8, a 2x4 cluster
     grid, two Erdos-Renyi random graphs, a Bell-pair matching graph, and
     the GHZ (star) graph -- each driven by explicit (m, k), each
     certified exactly, and each producing its own saved figure.


In [1]:

from __future__ import annotations
import itertools
import os
import random
from typing import Iterable, Optional, Sequence, Tuple

import numpy as np

import matplotlib
matplotlib.use("Agg")  # headless / script-safe backend, no plt.show() needed
import matplotlib.pyplot as plt
import networkx as nx

# Where publication-quality figures get saved (PDF + high-res PNG per graph).
# Same folder as the script/notebook itself -- no separate subfolder.
# __file__ isn't defined when running inside a Jupyter/IPython cell, so fall
# back to the current working directory in that case.
try:
    FIGURE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    FIGURE_DIR = os.getcwd()



In [2]:
# GF(2) linear algebra

def gf2_rank(mat: np.ndarray) -> int:
    """Rank of a 0/1 matrix over GF(2) via Gaussian elimination."""
    A = mat.copy().astype(np.uint8) % 2
    rows, cols = A.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for r in range(rank, rows):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        A[[rank, pivot]] = A[[pivot, rank]]
        for r in range(rows):
            if r != rank and A[r, col]:
                A[r, :] ^= A[rank, :]
        rank += 1
        if rank == rows:
            break
    return rank


def gf2_is_invertible(mat: np.ndarray) -> bool:
    """A square 0/1 matrix is invertible over GF(2) iff it has full rank."""
    n, m = mat.shape
    if n != m:
        raise ValueError("Matrix must be square to test invertibility.")
    return gf2_rank(mat) == n


# Graph representation & cut matrix

def adjacency_matrix(n: int, edges: Iterable[Tuple[int, int]]) -> np.ndarray:
    A = np.zeros((n, n), dtype=np.uint8)
    for u, v in edges:
        A[u, v] = 1
        A[v, u] = 1
    return A


def cut_matrix(A: np.ndarray, S: Sequence[int], N: Sequence[int]) -> np.ndarray:
    """Gamma_{S,N}: rows indexed by S, columns indexed by N."""
    return A[np.ix_(S, N)]



In [3]:
# --------------------------------------------------------------------------
# (m, k) parameter validation
# --------------------------------------------------------------------------

def validate_graph_size(A: np.ndarray, m: int, k: int) -> int:
    """
    Enforce |V(G)| = 2mk, the first step of GSECC. Returns mk on success;
    raises ValueError immediately on mismatch, per the problem statement:

        1. Verify |V| = 2mk. If not, reject immediately.
    """
    if m <= 0 or k <= 0:
        raise ValueError(f"m and k must be positive integers (got m={m}, k={k}).")
    mk = m * k
    n = A.shape[0]
    if n != 2 * mk:
        raise ValueError(
            f"Graph has {n} vertices, but expected 2*m*k = {2 * mk} "
            f"for m={m}, k={k}."
        )
    return mk


def print_parameters(m: int, k: int, A: np.ndarray) -> None:
    """Print the (m, k, mk, |V|) header identifying the encrypted-cloning
    instance being certified."""
    mk = m * k
    n = A.shape[0]
    print("Encrypted Cloning Parameters")
    print("----------------------------")
    print(f"m  = {m}")
    print(f"k  = {k}")
    print(f"mk = {mk}")
    print(f"Graph vertices = {n}")


# --------------------------------------------------------------------------
# GSECC: verification of a single candidate certificate (S, N)
# --------------------------------------------------------------------------

def verify_certificate(A: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int) -> bool:
    """
    Polynomial-time verifier for the GSECC certificate (S, N) at the given
    (m, k): does this balanced bipartition satisfy
    rank_GF(2)(Gamma_{S,N}) = mk (equivalently, is Gamma_{S,N} invertible
    over GF(2))?

    Complexity: O((mk)^3) via GF(2) Gaussian elimination.
    """
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    if len(S) != len(N) or len(S) + len(N) != n or len(S) != mk:
        raise ValueError(f"S, N must each have size mk = {mk} and partition all {n} vertices.")
    if set(S) & set(N):
        raise ValueError("S and N must be disjoint.")
    Gamma = cut_matrix(A, list(S), list(N))
    return gf2_is_invertible(Gamma)


# --------------------------------------------------------------------------
# GSECC: exhaustive search (exact, exponential -- fine for small n)
# --------------------------------------------------------------------------

def find_certificate_exact(A: np.ndarray, m: int, k: int) -> Optional[Tuple[Tuple[int, ...], Tuple[int, ...]]]:
    """
    The exact GSECC algorithm: search ALL balanced bipartitions of a
    2mk-vertex graph and return a witness (S, N) satisfying
    rank_GF(2)(Gamma_{S,N}) = mk as soon as one is found, else None.

    Because the search is exhaustive over every balanced partition, a
    return value of None is a mathematical PROOF that no valid
    certificate exists for this graph at this (m, k) -- not a heuristic
    failure. See report_certificate() for how this is surfaced.

    Complexity: O( C(2mk-1, mk-1) * (mk)^3 ). Exponential in mk, so this
    is exact but scales poorly for very large mk (see module docstring).
    """
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]
    vertices = range(n)
    others = [v for v in vertices if v != 0]
    for combo in itertools.combinations(others, mk - 1):
        S = (0,) + combo
        N = tuple(v for v in vertices if v not in S)
        if verify_certificate(A, S, N, m, k):
            return S, N
    return None


In [4]:

# Exact graph-state / GHZ / Bell-pair construction

def graph_state_from_adjacency(A: np.ndarray) -> np.ndarray:
    """Construct the graph state |G> = CZ_E |+>^{\\otimes n} (small n only)."""
    n = A.shape[0]
    plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
    psi = plus
    for _ in range(n - 1):
        psi = np.kron(psi, plus)
    psi = psi.reshape([2] * n)
    for i in range(n):
        for j in range(i + 1, n):
            if A[i, j]:
                for bits in itertools.product([0, 1], repeat=n):
                    if bits[i] == 1 and bits[j] == 1:
                        psi[bits] *= -1
    return psi.reshape(-1)


def ghz_state(n: int) -> np.ndarray:
    """|GHZ_n> = (|0...0> + |1...1>)/sqrt(2), as a length-2^n vector."""
    psi = np.zeros(2 ** n, dtype=complex)
    psi[0] = 1 / np.sqrt(2)
    psi[-1] = 1 / np.sqrt(2)
    return psi


def bell_pair_matching_graph(m: int, k: int) -> np.ndarray:
    """
    Perfect-matching graph on 2mk vertices: i <-> i+mk for i in [0, mk).
    Each edge, under CZ|++>, gives a two-qubit graph state locally
    equivalent to a Bell pair; the whole state is mk independent Bell
    pairs, one per (S-vertex, N-vertex) pair -- the canonical "Bell pair"
    encrypted-cloning resource.
    """
    mk = m * k
    edges = [(i, i + mk) for i in range(mk)]
    return adjacency_matrix(2 * mk, edges)


def star_graph(m: int, k: int) -> np.ndarray:
    """
    Star graph on 2mk vertices (vertex 0 connected to all others). This is
    the standard graph-state representative locally equivalent to the GHZ
    state -- used here as the "GHZ" graph example for GSECC.
    """
    mk = m * k
    n = 2 * mk
    edges = [(0, v) for v in range(1, n)]
    return adjacency_matrix(n, edges)


def path_graph(m: int, k: int) -> np.ndarray:
    """Linear cluster-state chain: 0-1-2-...-(2mk-1)."""
    mk = m * k
    n = 2 * mk
    edges = [(i, i + 1) for i in range(n - 1)]
    return adjacency_matrix(n, edges)


def grid_graph(rows: int, cols: int) -> np.ndarray:
    """2D cluster-state grid, rows x cols, row-major vertex labeling."""
    n = rows * cols
    edges = []
    for r in range(rows):
        for c in range(cols):
            v = r * cols + c
            if c + 1 < cols:
                edges.append((v, v + 1))
            if r + 1 < rows:
                edges.append((v, v + cols))
    return adjacency_matrix(n, edges)


def reduced_density_matrix(state: np.ndarray, keep: Sequence[int], n: int) -> np.ndarray:
    """Reduced density matrix on the qubits listed in keep."""
    keep = list(sorted(keep))
    trace = [i for i in range(n) if i not in keep]
    psi = state.reshape([2] * n)
    perm = keep + trace
    psi = np.transpose(psi, perm)
    dk = 2 ** len(keep)
    dt = 2 ** len(trace)
    psi = psi.reshape(dk, dt)
    return psi @ psi.conj().T


def rho_S_matches_maximally_mixed(rho_S: np.ndarray, m: int, k: int, atol: float = 1e-9):
    """
    Quantitative version of 'rho_S == I/2^{mk}': returns (matches: bool,
    residual_norm: float) where residual_norm = || rho_S - I/2^{mk} ||_F.
    No matrices are printed -- just the pass/fail flag and the norm.
    """
    mk = m * k
    target = np.eye(2 ** mk) / (2 ** mk)
    residual = np.linalg.norm(rho_S - target)
    return residual <= atol, residual


In [5]:

# Visualization

def plot_graph(
    A: np.ndarray,
    S: Optional[Sequence[int]] = None,
    N: Optional[Sequence[int]] = None,
    title: str = "Graph",
    save: bool = True,
    fig_dir: str = FIGURE_DIR,
) -> Optional[str]:
    """
    If S, N are given (a certified GSECC partition):
        - vertices are arranged in two columns, S on the left, N on the
          right, labeled S_1..S_mk and N_1..N_mk
        - cut edges (S-N, i.e. the entries of Gamma_{S,N}) are drawn in
          blue and bold
        - internal edges (S-S or N-N) are drawn in light gray

    Otherwise (no certificate found):
        - falls back to an ordinary spring-layout drawing of the graph.

    When save=True, writes both a vector PDF and a 600 dpi PNG into
    fig_dir, named after `title`, and returns the PDF path. Uses a
    non-interactive backend, so no window is opened.
    """
    G = nx.from_numpy_array(A)

    plt.figure(figsize=(8, 7))

    if S is None or N is None:
        # -------- ordinary graph (no certificate to visualize) ---------
        pos = nx.spring_layout(G, seed=5)
        nx.draw_networkx(
            G,
            pos,
            node_color="#4CAF50",
            node_size=700,
            edgecolors="black",
            linewidths=1.2,
            font_size=12,
            font_weight="bold",
        )
    else:
        S = list(S)
        N = list(N)
        mk = len(S)

        pos = {}
        y = list(range(mk))[::-1]

        for i, v in enumerate(S):
            pos[v] = (0, y[i])
        for i, v in enumerate(N):
            pos[v] = (4, y[i])

        labels = {}
        for i, v in enumerate(S):
            labels[v] = rf"$S_{{{i + 1}}}$"
        for i, v in enumerate(N):
            labels[v] = rf"$N_{{{i + 1}}}$"

        Sset = set(S)
        Nset = set(N)
        cut = []
        inside = []
        for u, v in G.edges():
            if (u in Sset and v in Nset) or (u in Nset and v in Sset):
                cut.append((u, v))
            else:
                inside.append((u, v))

        nx.draw_networkx_edges(G, pos, edgelist=inside, edge_color="0.75", width=1.5)
        nx.draw_networkx_edges(G, pos, edgelist=cut, edge_color="#1565C0", width=3)

        nx.draw_networkx_nodes(
            G, pos, nodelist=S, node_color="#43A047",
            edgecolors="black", linewidths=1.5, node_size=850,
        )
        nx.draw_networkx_nodes(
            G, pos, nodelist=N, node_color="#FB8C00",
            edgecolors="black", linewidths=1.5, node_size=850,
        )
        nx.draw_networkx_labels(G, pos, labels, font_size=14, font_weight="bold")

        plt.text(0, mk + 0.3, r"$S$", fontsize=18, ha="center")
        plt.text(4, mk + 0.3, r"$N$", fontsize=18, ha="center")

    plt.title(title, fontsize=18)
    plt.axis("off")
    plt.tight_layout()

    pdf_path = None
    if save:
        os.makedirs(fig_dir, exist_ok=True)
        filename = title
        for bad, good in [(" ", "_"), ("(", ""), (")", ""), ("/", "_"),
                           ("\\", "_"), (",", ""), ("=", ""), (":", "")]:
            filename = filename.replace(bad, good)
        pdf_path = os.path.join(fig_dir, filename + ".pdf")
        png_path = os.path.join(fig_dir, filename + ".png")
        plt.savefig(pdf_path, bbox_inches="tight")
        plt.savefig(png_path, dpi=600, bbox_inches="tight")

    plt.close()
    return pdf_path



In [6]:
# --------------------------------------------------------------------------
# GSDC: decoder construction
# --------------------------------------------------------------------------

def _coefficient_matrix(psi: np.ndarray, S: Sequence[int], N: Sequence[int], n: int, mk: int) -> np.ndarray:
    """
    Reorder qubits so S precedes N, then reshape |G> into the
    2^{mk} x 2^{mk} coefficient matrix M with |G> = sum_ij M_ij |i>_S|j>_N.
    """
    S = list(S)
    N = list(N)
    perm = S + N
    psi_t = np.transpose(psi.reshape([2] * n), perm)
    return psi_t.reshape(2 ** mk, 2 ** mk)


def construct_decoder(
    psi: np.ndarray, S: Sequence[int], N: Sequence[int], m: int, k: int, atol: float = 1e-9
):
    """
    GSDC: given the certified partition (S, N) for the instance (m, k),
    construct the noise-side unitary W such that
    |G> = (I_S ⊗ W) |Phi_{2^{mk}}>.

    Steps (all O(d^2) for d = 2^{mk}):
      1. M = coefficient matrix of |G> in the S-then-N qubit ordering.
      2. Q = sqrt(d) * M   (unitary, given rho_S = I/d).
      3. W = Q^T.

    Returns a dict with W, a unitarity check ||W^dagger W - I||, and a
    reconstruction check comparing (I_S ⊗ W)|Phi_d> against |G> (up to the
    S-then-N qubit reordering used throughout).
    """
    mk = m * k
    n = len(S) + len(N)
    d = 2 ** mk
    M = _coefficient_matrix(psi, S, N, n, mk)
    Q = np.sqrt(d) * M
    W = Q.T

    unitarity_residual = np.linalg.norm(W.conj().T @ W - np.eye(d))

    # Canonical maximally entangled state as a d x d matrix: Phi_mat = I/sqrt(d)
    Phi_mat = np.eye(d) / np.sqrt(d)
    # Apply (I_S ⊗ W): new_M[i, j'] = sum_j Phi_mat[i, j] * W[j', j]
    reconstructed_M = Phi_mat @ W.T

    reconstruction_residual = np.linalg.norm(reconstructed_M - M)

    return {
        "W": W,
        "unitarity_residual": unitarity_residual,
        "is_unitary": unitarity_residual <= atol,
        "reconstruction_residual": reconstruction_residual,
        "reconstructs_state": reconstruction_residual <= atol,
    }


# --------------------------------------------------------------------------
# Shared reporting routine
# --------------------------------------------------------------------------

def report_certificate(
    label: str,
    A: np.ndarray,
    m: int,
    k: int,
    result: Optional[Tuple[Sequence[int], Sequence[int]]],
    build_state: bool = True,
    max_mk_for_state: int = 6,
    state_override: Optional[np.ndarray] = None,
) -> None:
    """
    Report a GSECC outcome for the instance (m, k), and -- whenever a
    certificate exists and the state is small enough to build explicitly
    -- run GSDC to construct and verify the decoder W.

    rho_S is never printed as a matrix; only the boolean
    'rho_S == I/2^{mk}' check and its residual norm are shown.

    `result` is expected to come from find_certificate_exact(), which is
    exhaustive over every balanced partition. Consequently a None result
    is always reported as a proven negative -- "No valid balanced
    partition exists" -- rather than as an inconclusive search outcome.

    state_override lets callers supply a state vector that is *not*
    generated from A via CZ gates (e.g. an actual GHZ state), while still
    routing through the same reporting / rho_S-check / GSDC logic.
    """
    mk = validate_graph_size(A, m, k)
    n = A.shape[0]

    print_parameters(m, k, A)
    print(f"Graph: {label}")

    if result is None:
        fig_path = plot_graph(A, title=label)
        print("  No valid balanced partition exists.")
        print("  Graph is NOT a valid encrypted cloning resource.")
        if fig_path:
            print(f"  Figure saved: {fig_path}")
        print()
        return

    S, N = result
    fig_path = plot_graph(A, S, N, title=label)
    print(f"  Certified! S = {set(S)}; N = {set(N)}")
    if fig_path:
        print(f"  Figure saved: {fig_path}")

    Gamma = cut_matrix(A, list(S), list(N))
    rank = gf2_rank(Gamma)
    print(f"  GF(2) rank(Gamma_S,N) = {rank}  (need {mk})  -> "
          f"{'PASS' if rank == mk else 'FAIL'}")

    if build_state and mk <= max_mk_for_state:
        psi = state_override if state_override is not None else graph_state_from_adjacency(A)
        rho_S = reduced_density_matrix(psi, list(S), n)
        matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
        print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})")

        dec = construct_decoder(psi, S, N, m, k)
        print(f"  GSDC: W unitary : {dec['is_unitary']}   "
              f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
        print(f"  GSDC: |G> == (I_S ⊗ W)|Phi_{{{2 ** mk}}}> : "
              f"{dec['reconstructs_state']}   "
              f"(residual = {dec['reconstruction_residual']:.3e})")
    elif build_state:
        print(f"  (skipping explicit state construction: mk = {mk} > "
              f"max_mk_for_state = {max_mk_for_state}, state vector would "
              f"have 2^{n} = {2 ** n} amplitudes)")

    print()


def report_non_graph_state(
    label: str, psi: np.ndarray, n: int, m: int, k: int, S: Sequence[int], N: Sequence[int]
) -> None:
    """
    Reporting routine for states not obtained from a certified GSECC graph
    search (e.g. GHZ tested against an arbitrary balanced bipartition).
    """
    mk = m * k
    print_parameters(m, k, np.zeros((n, n), dtype=np.uint8))
    print(f"State: {label}  (S = {set(S)}, N = {set(N)})")
    rho_S = reduced_density_matrix(psi, list(S), n)
    matches, residual = rho_S_matches_maximally_mixed(rho_S, m, k)
    print(f"  rho_S == I/2^mk : {matches}   (||rho_S - I/2^mk||_F = {residual:.3e})  "
          f"-> {'valid encrypted-cloning resource' if matches else 'NOT a valid resource'}")

    dec = construct_decoder(psi, S, N, m, k)
    print(f"  GSDC: W unitary : {dec['is_unitary']}   "
          f"(||W^dagger W - I||_F = {dec['unitarity_residual']:.3e})")
    print()



In [8]:
print("=== GSECC certification + GSDC decoder construction ===\n")
print(f"Figures will be saved under: {FIGURE_DIR}\n")

# Common parameters for all examples
m, k = 3, 4

# -- Example 1: complete graph ---------------------------------------------
A = adjacency_matrix(2 * m * k, itertools.combinations(range(2 * m * k), 2))
result = find_certificate_exact(A, m, k)
report_certificate("Complete graph", A, m, k, result)

# -- Example 2: cycle graph ------------------------------------------------
A = adjacency_matrix(
    2 * m * k,
    [(i, (i + 1) % (2 * m * k)) for i in range(2 * m * k)]
)
result = find_certificate_exact(A, m, k)
report_certificate("Cycle graph", A, m, k, result)

# -- Example 3: linear cluster chain ---------------------------------------
A = path_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("Path / linear cluster chain", A, m, k, result)

# -- Example 4: 2x4 cluster grid -------------------------------------------
rows, cols = 2, 2 * m * k // 2   # choose dimensions with rows*cols = 2mk
A = grid_graph(rows, cols)
result = find_certificate_exact(A, m, k)
report_certificate(f"{rows}x{cols} cluster grid", A, m, k, result)

# -- Example 5: dense Erdos-Renyi random graph ------------------------------
n = 2 * m * k
p = 0.9
rng = random.Random(42)
edges = [(i, j) for i in range(n)
         for j in range(i + 1, n)
         if rng.random() < p]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    f"G(n={n}, p={p}) dense random graph",
    A, m, k, result,
    build_state=False
)

# -- Example 6: sparse Erdos-Renyi random graph -----------------------------
n = 2 * m * k
p = 0.35
rng = random.Random(7)
edges = [(i, j) for i in range(n)
         for j in range(i + 1, n)
         if rng.random() < p]
A = adjacency_matrix(n, edges)
result = find_certificate_exact(A, m, k)
report_certificate(
    f"G(n={n}, p={p}) sparse random graph",
    A, m, k, result,
    build_state=True,
    max_mk_for_state=6
)

# -- Example 7: Bell-pair matching graph -----------------------------------
A = bell_pair_matching_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate(
    f"Bell-pair matching graph ({m * k} Bell pairs)",
    A, m, k, result
)

# -- Example 8: GHZ / star graph -------------------------------------------
A = star_graph(m, k)
result = find_certificate_exact(A, m, k)
report_certificate("GHZ (star graph)", A, m, k, result)

# -- Example 8b: actual GHZ state vector -----------------------------------
n = 2 * m * k
psi_ghz = ghz_state(n)
mk = m * k
S_arbitrary = tuple(range(mk))
N_arbitrary = tuple(range(mk, n))
report_non_graph_state(
    "GHZ state vector (arbitrary balanced cut)",
    psi_ghz,
    n,
    m,
    k,
    S_arbitrary,
    N_arbitrary
)

# -- Example 9: mismatched (m, k) ------------------------------------------
print("Deliberately mismatched (m, k) example:")
try:
    A_bad = adjacency_matrix(
        10,
        itertools.combinations(range(10), 2)
    )  # intentionally wrong size
    find_certificate_exact(A_bad, m, k)
except ValueError as e:
    print(f"  Rejected as expected: {e}")
print()

if os.path.isdir(FIGURE_DIR):
    saved = sorted(f for f in os.listdir(FIGURE_DIR) if f.endswith(".pdf"))
    print(f"Saved {len(saved)} figures (PDF + 600 dpi PNG each) to {FIGURE_DIR}:")
    for f in saved:
        print(f"  {f}")

=== GSECC certification + GSDC decoder construction ===

Figures will be saved under: C:\Users\Dell\Downloads\graph state check

Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Graph: Complete graph
  No valid balanced partition exists.
  Graph is NOT a valid encrypted cloning resource.
  Figure saved: C:\Users\Dell\Downloads\graph state check\Complete_graph.pdf

Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Graph: Cycle graph
  Certified! S = {0, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21}; N = {2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 23}
  Figure saved: C:\Users\Dell\Downloads\graph state check\Cycle_graph.pdf
  GF(2) rank(Gamma_S,N) = 12  (need 12)  -> PASS
  (skipping explicit state construction: mk = 12 > max_mk_for_state = 6, state vector would have 2^24 = 16777216 amplitudes)

Encrypted Cloning Parameters
----------------------------
m  = 3
k  = 4
mk = 12
Graph vertices = 24
Grap